# 03 — RetinaNet (One-stage Anchor-based Detector)

**Experiment:** COCO-pretrained RetinaNet ResNet-50 FPN v2, 3-fold video-level cross-validation, frame step 6, batch 2, 6 epochs, SGD. Original train images only; original validation images are reserved as holdout.

The same `cv_splits_step6.csv` used by Faster R-CNN is loaded **from Google Drive**. Each fold starts from independent COCO-pretrained weights, saves a resumable checkpoint after each epoch, and evaluates COCO mAP/AP50/AP_small on its own validation fold. This is a **baseline**, not a tuned or ignore-aware model.

> **Before Run all:** Runtime → Change runtime type → **T4 GPU** (or another CUDA GPU). Confirm that `MyDrive/UCU_AI_HW2/visdrone_competition.zip` and `cv_splits_step6.csv` exist. **No Kaggle API token or kaggle.json required.**

## 1 — Environment and Drive
Do not reinstall PyTorch/torchvision: use the compatible packages preinstalled in Colab. Only install COCO evaluation if missing. No warnings are hidden.

In [1]:
%pip -q install pycocotools

import gc
import json
import math
import os
import random
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from collections import defaultdict
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import pil_to_tensor
from google.colab import drive
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

drive.mount('/content/drive')

SEED = 42
FRAME_STEP = 6
N_FOLDS = 3
BATCH_SIZE = 2
EPOCHS = 6
MIN_SIZE = 640
MAX_SIZE = 1024
NUM_WORKERS = 0  # avoids Colab multiprocessing teardown errors

ROOT = Path('/content/drive/MyDrive/UCU_AI_HW2')
ARCHIVE_PATH = ROOT / 'visdrone_competition.zip'
SPLIT_PATH = ROOT / 'cv_splits_step6.csv'
RESULTS_ROOT = ROOT / 'retinanet_v2_baseline'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'Select a GPU runtime (Runtime > Change runtime type).'
DEVICE = torch.device('cuda')
assert ARCHIVE_PATH.is_file(), f'Missing existing dataset archive: {ARCHIVE_PATH}'
assert SPLIT_PATH.is_file(), f'Missing exact Faster R-CNN folds: {SPLIT_PATH}'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Dataset archive:', ARCHIVE_PATH)
print('CV split:', SPLIT_PATH)
print('Checkpoint root:', RESULTS_ROOT)

Mounted at /content/drive
Torch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
GPU: Tesla T4
Dataset archive: /content/drive/MyDrive/UCU_AI_HW2/visdrone_competition.zip
CV split: /content/drive/MyDrive/UCU_AI_HW2/cv_splits_step6.csv
Checkpoint root: /content/drive/MyDrive/UCU_AI_HW2/retinanet_v2_baseline


## 2 — Extract the shared dataset to local Colab storage
The ZIP remains on Google Drive. Each new Colab runtime must extract the images locally again. The extraction is skipped when the complete local train and validation folders are already present.

In [2]:
LOCAL_DIR = Path('/content/visdrone')
LOCAL_DIR.mkdir(parents=True, exist_ok=True)


def find_root(base):
    for ann_file in base.rglob('train/annotations_coco.json'):
        root = ann_file.parent.parent
        if (root / 'train/images').is_dir() and (root / 'val/annotations_coco.json').is_file():
            return root
    return None

DATA_ROOT = find_root(LOCAL_DIR)

if DATA_ROOT is None:
    assert zipfile.is_zipfile(ARCHIVE_PATH), 'Drive ZIP is invalid/incomplete.'
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        members = archive.infolist()
        required = sum(m.file_size for m in members)
        available = shutil.disk_usage('/content').free
        print(f'Uncompressed size: {required/1e9:.2f} GB; free: {available/1e9:.2f} GB')
        assert available > required * 1.10, 'Insufficient free local disk for the dataset.'
        # ZIP is an input from the competition: also reject paths escaping the extraction root.
        base = LOCAL_DIR.resolve()
        for member in members:
            destination = (LOCAL_DIR / member.filename).resolve()
            assert destination == base or base in destination.parents, f'Unsafe ZIP member: {member.filename}'
        print('Extracting dataset locally...')
        archive.extractall(LOCAL_DIR)
    DATA_ROOT = find_root(LOCAL_DIR)

assert DATA_ROOT is not None, 'Expected train/annotations_coco.json + train/images not found.'

with (DATA_ROOT / 'train/annotations_coco.json').open('r') as f:
    train_coco = json.load(f)
with (DATA_ROOT / 'val/annotations_coco.json').open('r') as f:
    holdout_coco = json.load(f)

assert len(train_coco['images']) == 24201, 'Unexpected train image count.'
assert len(holdout_coco['images']) == 2846, 'Unexpected untouched holdout size.'
print('Dataset root:', DATA_ROOT)
print('Train images:', len(train_coco['images']))
print('Original holdout images:', len(holdout_coco['images']))

Uncompressed size: 10.98 GB; free: 120.80 GB
Extracting dataset locally...
Dataset root: /content/visdrone
Train images: 24201
Original holdout images: 2846


## 3 — Load the **existing** exact sequence-level folds
Never regenerate folds independently: use the same `image_id → fold` assignment saved by Faster R-CNN.

In [4]:
sampled = pd.read_csv(SPLIT_PATH)
required_columns = {'image_id', 'fold', 'clip'}
assert required_columns.issubset(sampled.columns), f'Missing columns: {required_columns - set(sampled.columns)}'
assert len(sampled) == 4062, f'Expected 4062 sampled images, got {len(sampled)}'
assert sampled['image_id'].is_unique
assert set(sampled['fold'].astype(int).unique()) == {0, 1, 2}
assert sampled['fold'].notna().all()

original_image_ids = {int(img['id']) for img in train_coco['images']}
assert set(sampled['image_id'].astype(int)).issubset(original_image_ids)

for fold in range(3):
    train_part = sampled.loc[sampled['fold'] != fold]
    val_part = sampled.loc[sampled['fold'] == fold]
    assert set(train_part['clip']).isdisjoint(set(val_part['clip'])), 'Video leakage detected!'
    print(
        f"Fold {fold}: "
        f"train={len(train_part)}, "
        f"val={len(val_part)}, "
        f"train clips={train_part['clip'].nunique()}, "
        f"val clips={val_part['clip'].nunique()}"
    )


assert sorted(sampled['fold'].value_counts().tolist()) == [1349, 1356, 1357]
print('Existing 3-fold split verified.')

Fold 0: train=2705, val=1357, train clips=37, val clips=19
Fold 1: train=2706, val=1356, train clips=37, val clips=19
Fold 2: train=2713, val=1349, train clips=38, val clips=18
Existing 3-fold split verified.


## 4 — VisDrone PyTorch Dataset
We use **0-based training labels (0–7)** for RetinaNet's eight sigmoid classification channels. During COCO evaluation, predicted categories are converted back to **1–8**. `iscrowd=1` annotations are excluded **only from training targets**; original COCO annotations are retained during evaluation. This is not an ignore-aware training implementation.

In [5]:
image_info = {int(img['id']): img for img in train_coco['images']}
annotations_by_image = defaultdict(list)
for ann in train_coco['annotations']:
    annotations_by_image[int(ann['image_id'])].append(ann)


def image_path_from_info(info):
    rel = Path(info['file_name'])
    candidates = (
        DATA_ROOT / 'train' / rel,
        DATA_ROOT / 'train/images' / rel,
        DATA_ROOT / 'train/images' / rel.parent.name / rel.name,
        DATA_ROOT / rel,
    )
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(f'Image not found for {info["file_name"]}; checked {candidates}')


class VisDroneRetinaDataset(Dataset):
    def __init__(self, image_ids):
        self.ids = sorted(int(x) for x in image_ids)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        with Image.open(image_path_from_info(image_info[image_id])) as im:
            image = im.convert('RGB')
            width, height = image.size
            image_tensor = pil_to_tensor(image).float().div_(255.0)

        boxes, labels, areas = [], [], []
        for ann in annotations_by_image[image_id]:
            if int(ann.get('iscrowd', 0)) == 1:
                continue
            category_id = int(ann['category_id'])
            if category_id not in range(1, 9):
                continue
            x, y, w, h = map(float, ann['bbox'])
            x1 = min(width, max(0.0, x))
            y1 = min(height, max(0.0, y))
            x2 = min(width, max(0.0, x + w))
            y2 = min(height, max(0.0, y + h))
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append((x1, y1, x2, y2))
            labels.append(category_id - 1)  # RetinaNet logits use 0..7
            areas.append((x2 - x1) * (y2 - y1))

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            'labels': torch.tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor(image_id, dtype=torch.int64),
            'area': torch.tensor(areas, dtype=torch.float32),
            'iscrowd': torch.zeros(len(labels), dtype=torch.int64),
        }
        return image_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


def make_fold_loaders(fold):
    train_ids = sampled.loc[sampled['fold'] != fold, 'image_id']
    val_ids = sampled.loc[sampled['fold'] == fold, 'image_id']
    train_dataset = VisDroneRetinaDataset(train_ids)
    val_dataset = VisDroneRetinaDataset(val_ids)
    train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn,
                          pin_memory=True)
    val_dl = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, collate_fn=collate_fn,
                        pin_memory=True)
    return train_dl, val_dl, set(int(x) for x in val_ids)

train_check, val_check, _ = make_fold_loaders(0)
images_check, targets_check = next(iter(train_check))
assert len(images_check) == BATCH_SIZE
assert all(torch.isfinite(t['boxes']).all().item() for t in targets_check)
assert all(((t['labels'] >= 0) & (t['labels'] < 8)).all().item() for t in targets_check)
print('Input tensor:', tuple(images_check[0].shape))
print('Boxes in first image:', len(targets_check[0]['boxes']))
print('Dataset / label smoke test passed.')
del train_check, val_check, images_check, targets_check

Input tensor: (3, 1071, 1904)
Boxes in first image: 44
Dataset / label smoke test passed.


## 5 — Model: COCO-pretrained RetinaNet ResNet-50 FPN v2
The pretrained GroupNorm classification tower and box regression head are preserved; only the final classification logits layer is replaced for eight VisDrone classes. The model is **anchor-based** with its default anchors. RetinaNet v2 uses Focal Loss for classification and GIoU regression loss by default. <https://docs.pytorch.org/vision/main/models.html>

In [6]:
from torchvision.models.detection import (
    retinanet_resnet50_fpn_v2,
    RetinaNet_ResNet50_FPN_V2_Weights,
)


def build_model():
    model = retinanet_resnet50_fpn_v2(
        weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT,
        min_size=MIN_SIZE,
        max_size=MAX_SIZE,
        score_thresh=0.01,
        nms_thresh=0.5,
        detections_per_img=300,
        topk_candidates=1000,
    )
    cls_head = model.head.classification_head
    old_logits = cls_head.cls_logits
    anchors_per_location = model.anchor_generator.num_anchors_per_location()
    assert len(set(anchors_per_location)) == 1
    n_anchors = anchors_per_location[0]
    cls_head.cls_logits = torch.nn.Conv2d(
        in_channels=old_logits.in_channels,
        out_channels=n_anchors * 8,
        kernel_size=3,
        stride=1,
        padding=1,
    )
    torch.nn.init.normal_(cls_head.cls_logits.weight, std=0.01)
    torch.nn.init.constant_(cls_head.cls_logits.bias, -math.log(99.0))
    cls_head.num_classes = 8
    return model.to(DEVICE)


def make_optimizer(model):
    return torch.optim.SGD(model.parameters(), lr=0.002,
                           momentum=0.9, weight_decay=0.0005)

print('RetinaNet model factory configured.')

RetinaNet model factory configured.


## 6 — One-batch **forward + backward** smoke test
This checks CUDA memory and gradient computation before any multi-hour training. If `CUDA out of memory` occurs here, stop: do not start training or silently change the batch size. The 3-fold setup is designed for batch size 2.

In [7]:
gc.collect()
torch.cuda.empty_cache()
smoke_model = build_model()
smoke_model.train()
smoke_loader, _, _ = make_fold_loaders(0)
smoke_images, smoke_targets = next(iter(smoke_loader))
smoke_images = [im.to(DEVICE) for im in smoke_images]
smoke_targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in smoke_targets]

smoke_model.zero_grad(set_to_none=True)
with torch.autocast(device_type='cuda', dtype=torch.float16):
    smoke_losses = smoke_model(smoke_images, smoke_targets)
    smoke_total = sum(smoke_losses.values())
assert bool(torch.isfinite(smoke_total).item()), 'Non-finite loss in smoke test.'
smoke_scaler = torch.amp.GradScaler('cuda')
smoke_scaler.scale(smoke_total).backward()

for key, value in smoke_losses.items():
    print(f'{key}: {value.item():.5f}')
print(f'Total: {smoke_total.item():.5f}')
print(f'Peak CUDA allocation: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')
print('Forward + backward smoke test passed.')

del smoke_model, smoke_loader, smoke_images, smoke_targets, smoke_losses, smoke_total, smoke_scaler
gc.collect()
torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth


100%|██████████| 146M/146M [00:01<00:00, 152MB/s]


classification: 1.14051
bbox_regression: 0.25476
Total: 1.39527
Peak CUDA allocation: 1.54 GB
Forward + backward smoke test passed.


## 7 — COCO evaluation function
Standard COCO AP uses max detections `[1, 10, 100]`. Original `iscrowd` annotations are included in ground truth, and predictions map class `0–7` to category `1–8`. This matches the evaluation approach used in the Faster R-CNN Colab notebook.

In [8]:
def evaluate_model(model, val_loader, val_image_ids, fold):
    model.eval()
    gt = {
        'images': [im for im in train_coco['images'] if int(im['id']) in val_image_ids],
        'annotations': [a for a in train_coco['annotations']
                        if int(a['image_id']) in val_image_ids and 1 <= int(a['category_id']) <= 8],
        'categories': [c for c in train_coco['categories'] if 1 <= int(c['id']) <= 8],
        'info': train_coco.get('info', {}),
        'licenses': train_coco.get('licenses', []),
    }
    coco_gt = COCO()
    coco_gt.dataset = gt
    coco_gt.createIndex()

    predictions = []
    inference_times = []
    with torch.inference_mode():
        for images, targets in tqdm(val_loader, desc=f'Fold {fold}: validation', leave=False):
            images_device = [im.to(DEVICE) for im in images]
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            outputs = model(images_device)
            torch.cuda.synchronize()
            inference_times.append((time.perf_counter() - t0) / len(images))

            for out, target in zip(outputs, targets):
                image_id = int(target['image_id'].item())
                for box, conf, cls in zip(
                    out['boxes'].detach().cpu().tolist(),
                    out['scores'].detach().cpu().tolist(),
                    out['labels'].detach().cpu().tolist(),
                ):
                    if not (0 <= int(cls) < 8) or conf < 0.01:
                        continue
                    x1, y1, x2, y2 = box
                    w, h = x2 - x1, y2 - y1
                    if w <= 0 or h <= 0:
                        continue
                    predictions.append({
                        'image_id': image_id,
                        'category_id': int(cls) + 1,
                        'bbox': [float(x1), float(y1), float(w), float(h)],
                        'score': float(conf),
                    })

    if not predictions:
        print(f'Fold {fold}: zero predictions; COCO AP metrics set to 0.')
        stats = np.zeros(12, dtype=float)
    else:
        coco_dt = coco_gt.loadRes(predictions)
        evaluator = COCOeval(coco_gt, coco_dt, iouType='bbox')
        evaluator.params.imgIds = sorted(val_image_ids)
        evaluator.params.catIds = list(range(1, 9))
        evaluator.params.maxDets = [1, 10, 100]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
        stats = evaluator.stats

    return {
        'model': 'RetinaNet ResNet50 FPN v2',
        'fold': fold,
        'epochs_completed': EPOCHS,
        'mAP50-95': float(stats[0]),
        'AP50': float(stats[1]),
        'AP75': float(stats[2]),
        'AP_small': float(stats[3]),
        'AP_medium': float(stats[4]),
        'AP_large': float(stats[5]),
        'inference_ms_per_frame': float(np.mean(inference_times) * 1000),
    }

print('COCO evaluator ready.')

COCO evaluator ready.


## 8 — Train all three folds; checkpoint and resume
This is the **main GPU training cell**. Each fold starts from the same pretrained weights, uses an independent optimizer/scheduler, and saves `latest.pt` at the end of every completed epoch. On a rerun, completed folds are skipped, interrupted folds resume from the last complete epoch. No exceptions or warnings are suppressed.

In [9]:
def save_checkpoint_safely(payload, folder, fold):
    # Write locally first to avoid a half-written Drive checkpoint.
    local = Path(f'/content/retinanet_fold_{fold}_latest.pt')
    partial = folder / 'latest_uploading.pt'
    final = folder / 'latest.pt'
    torch.save(payload, local)
    shutil.copy2(local, partial)
    partial.replace(final)


for fold in range(N_FOLDS):
    fold_dir = RESULTS_ROOT / f'fold_{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = fold_dir / 'latest.pt'
    metrics_path = fold_dir / 'validation_metrics.csv'
    history_path = fold_dir / 'history.csv'

    if metrics_path.is_file():
        completed = pd.read_csv(metrics_path)
        if (len(completed) == 1 and
                int(completed.iloc[0]['epochs_completed']) == EPOCHS and
                ckpt_path.is_file()):
            print(f'Fold {fold}: already completed; skipping.')
            continue

    # Independent seed / fresh COCO-pretrained model for each fold.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    gc.collect()
    torch.cuda.empty_cache()

    train_loader, val_loader, val_image_ids = make_fold_loaders(fold)
    model = build_model()
    optimizer = make_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda')
    history = []
    start_epoch = 0

    if ckpt_path.is_file():
        state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
        assert state['fold'] == fold and state['seed'] == SEED
        assert state['epochs'] == EPOCHS
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        start_epoch = int(state['epoch']) + 1
        history = state['history']
        del state
        print(f'Fold {fold}: resuming from epoch {start_epoch+1}.')
    else:
        print(f'Fold {fold}: starting fresh.')

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        epoch_total = 0.0
        epoch_cls = 0.0
        epoch_box = 0.0
        start = time.perf_counter()
        progress = tqdm(train_loader, desc=f'Fold {fold} · epoch {epoch+1}/{EPOCHS}', leave=False)

        for images, targets in progress:
            images = [im.to(DEVICE, non_blocking=True) for im in images]
            targets = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                losses = model(images, targets)
                total_loss = sum(losses.values())

            if not bool(torch.isfinite(total_loss).item()):
                raise FloatingPointError(f'Non-finite loss at fold {fold}, epoch {epoch+1}.')
            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_total += total_loss.item()
            epoch_cls += losses['classification'].item()
            epoch_box += losses['bbox_regression'].item()
            progress.set_postfix(loss=f'{total_loss.item():.4f}')

        scheduler.step()
        record = {
            'fold': fold,
            'epoch': epoch + 1,
            'train_loss': epoch_total / len(train_loader),
            'classification_loss': epoch_cls / len(train_loader),
            'bbox_regression_loss': epoch_box / len(train_loader),
            'learning_rate_next_epoch': optimizer.param_groups[0]['lr'],
            'minutes': (time.perf_counter() - start) / 60,
        }
        history.append(record)
        save_checkpoint_safely({
            'epoch': epoch,
            'fold': fold,
            'epochs': EPOCHS,
            'seed': SEED,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict(),
            'history': history,
        }, fold_dir, fold)
        pd.DataFrame(history).to_csv(history_path, index=False)
        print(f'Fold {fold} | epoch {epoch+1}/{EPOCHS} | '
              f'loss={record["train_loss"]:.4f} | {record["minutes"]:.1f} min | checkpoint saved')

    print(f'Fold {fold}: running COCO validation...')
    result = evaluate_model(model, val_loader, val_image_ids, fold)
    pd.DataFrame([result]).to_csv(metrics_path, index=False)
    print(f'Fold {fold} complete: mAP50-95={result["mAP50-95"]:.4f}, '
          f'AP50={result["AP50"]:.4f}, AP_small={result["AP_small"]:.4f}')

    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

print('All requested folds completed.')

Fold 0: starting fresh.


Fold 0 · epoch 1/6:   0%|          | 0/1353 [00:00<?, ?it/s]

Fold 0 | epoch 1/6 | loss=0.6828 | 7.7 min | checkpoint saved


Fold 0 · epoch 2/6:   0%|          | 0/1353 [00:00<?, ?it/s]

Fold 0 | epoch 2/6 | loss=0.5064 | 7.6 min | checkpoint saved


Fold 0 · epoch 3/6:   0%|          | 0/1353 [00:00<?, ?it/s]

... (33 outputs left out)

Fold 2 · epoch 6/6:   0%|          | 0/1357 [00:00<?, ?it/s]

Fold 2 | epoch 6/6 | loss=0.4445 | 7.5 min | checkpoint saved
Fold 2: running COCO validation...
creating index...
index created!


Fold 2: validation:   0%|          | 0/675 [00:00<?, ?it/s]

Loading and preparing results...
DONE (t=1.66s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=95.41s).
Accumulating evaluation results...
DONE (t=4.94s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.149
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.290
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.142
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.034
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.150
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.287
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.110
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.229
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.279
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=1

## 9 — Aggregate three-fold results
Only after all folds have `validation_metrics.csv` does the notebook calculate mean/std and save CSVs to Google Drive.

In [10]:
fold_frames = []
for fold in range(3):
    path = RESULTS_ROOT / f'fold_{fold}' / 'validation_metrics.csv'
    assert path.is_file(), f'Missing results for fold {fold}: {path}'
    fold_frames.append(pd.read_csv(path))

cv_results = pd.concat(fold_frames, ignore_index=True).sort_values('fold')
assert cv_results['fold'].tolist() == [0, 1, 2]

metric_cols = ['mAP50-95', 'AP50', 'AP75', 'AP_small',
               'AP_medium', 'AP_large', 'inference_ms_per_frame']
cv_summary = cv_results[metric_cols].agg(['mean', 'std']).T
cv_results.to_csv(RESULTS_ROOT / 'cv_3fold_results.csv', index=False)
cv_summary.to_csv(RESULTS_ROOT / 'cv_3fold_summary.csv')

print('Per-fold validation metrics:')
display(cv_results)
print('Mean and standard deviation:')
display(cv_summary.round(4))
print('Saved to:', RESULTS_ROOT)

Per-fold validation metrics:


,model,fold,epochs_completed,mAP50-95,AP50,AP75,AP_small,AP_medium,AP_large,inference_ms_per_frame
0,RetinaNet ResNet50 FPN v2,0,6,0.159505,0.302194,0.150626,0.034571,0.130138,0.337581,111.121303
1,RetinaNet ResNet50 FPN v2,1,6,0.141034,0.264064,0.129775,0.027300,0.131612,0.302537,111.037448
2,RetinaNet ResNet50 FPN v2,2,6,0.148736,0.289794,0.142115,0.034430,0.150000,0.287387,110.843954


Mean and standard deviation:


,mean,std
mAP50-95,0.1498,0.0093
AP50,0.2854,0.0194
AP75,0.1408,0.0105
AP_small,0.0321,0.0042
AP_medium,0.1373,0.0111
AP_large,0.3092,0.0257
inference_ms_per_frame,111.0009,0.1422


Saved to: /content/drive/MyDrive/UCU_AI_HW2/retinanet_v2_baseline
